# 라이브러리 및 함수

In [ ]:
# 필수 라이브러리 로드
import json
import re
import pandas as pd
from pathlib import Path

import os
from dotenv import load_dotenv
import openai
from openai import OpenAI
import jinja2

# 환경 변수에서 API 키 로드
load_dotenv()
openai.api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI()


In [ ]:
# 필터링된 JSONL 파일 경로
filtered_path = Path('filtered_five_or_more_N.jsonl')

# JSONL 데이터를 리스트로 읽어들임
drugs = []
if filtered_path.exists():
    with filtered_path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                drugs.append(json.loads(line))

print(f'Loaded {len(drugs)} drugs from {filtered_path}')


In [ ]:
system_prompt='''
당신은 의약품 정보를 기반으로 객관식 문제를 만드는 교육용 AI입니다. 
입력으로 주어지는 각 의약품에 대해 10개의 객관식 문제를 생성해야 합니다.

### 데이터 범위
- 제공된 정보를 넘어선 추측이나 임의의 지식은 추가하지 않습니다. 
- 할루시네이션을 피하기 위해 입력된 데이터에 없는 내용은 문제와 선택지에 절대 사용하지 않습니다.

### 문제 유형
- 각 문제는 "Which of the following "으로 시작해야 하며, 바로 다음으로는 is 또는 are를 씁니다.
- 이후 다음 템플릿 중 하나를 사용해 문제를 작성합니다. 문장 끝에는 "?"로 끝나야 합니다. 반드시 템플릿 형식을 준수하며, 불필요한 단어는 추가하지 않습니다.
    1. “the … mechanism …” (e.g. “the mechanism of action of X”)
    2. “the … explanation …” (e.g. “the explanation of how X works”)
    3. “the most specific etiology …” or “the … etiology …”
    4. “the … cause of/for …”
    5. “the … characteristic … that is causing/responsible for …”
    6. “most likely causing/responsible for …”
    7. “associated with …” or “consistent with …”
    8. “contributed to …”
    9. “the … cause …”
    10. “involved in …”
    11. “prevented by, reducing, or improve …”
    12. “administered to, added to, used to treat, or prescribed to …”
    13. “affected by …”
    14. “best represents/describes/explains/accounts for …”
    15. “contributed to …”
- 10개의 질문이 **서로 다른 주제를 다루도록** 합니다. 동일한 문항을 반복하지 않습니다.

### 선택지와 정답
- 각 문제는 4개의 선택지를 갖습니다. 
- 선택지는 제공된 데이터에서만 가져와서 구성합니다. 적절한 오답은 실제 데이터의 다른 필드를 이용하여 plausible하게 만듭니다.
- 문제당 정답은 단 1개이며, 중복정답을 허용하지 않습니다. 정답에는 `[correct]`, 오답에는 `[incorrect]`를 붙입니다.
- 학생들이 한 번호로 응답했을 때 고득점을 얻는 것을 막기 위해, 전체 문제에서 정답 선택지는 A, B, C, D 중 한쪽에 몰리지 않도록 균등한 비율로 분배합니다.
- 모든 내용은 영어로 작성합니다.

### 출력 형식
- 각 문제에 대해 질문(Question: …)과 선택지 A–D를 한 줄씩 작성합니다. 
- 마지막에 `Answer:` 항목을 추가하여 정답 선택지의 라벨(A/B/C/D)을 표기합니다.
- 10개의 문제를 연속해서 생성하되, 각 문제는 위와 같은 형식을 갖춰야 합니다.
'''

user_prompt_template = '''
당신에게 제공된 의약품 정보는 다음과 같습니다:
- Name: {{ name }}
{% if description %}- Description: {{ description }}{% endif %}
{% if mechanism_of_action %}- Mechanism of action: {{ mechanism_of_action }}{% endif %}
{% if pharmacodynamics %}- Pharmacodynamics: {{ pharmacodynamics }}{% endif %}
{% if indication %}- Indication: {{ indication }}{% endif %}
{% if classification %}- Classification: {{ classification }}{% endif %}
{% if affected_organisms %}- Affected organisms: {{ affected_organisms }}{% endif %}
{% if toxicity %}- Toxicity: {{ toxicity }}{% endif %}


위 정보를 사용하여 이 약물에 관한 객관식 문제 10개를 영어로 만들어 주세요.
각 문제는 서로 다른 패턴을 사용하여 약물의 기전, 설명, 병인, 영향받는 생명체, 상호작용 등 다양한 측면을 묻도록 구성하십시오.
10개의 문제의 정답 라벨링은 균등하게 분배합니다. 예를 들어 10개 문제의 정답이 모두 A여서는 안됩니다.
반드시 다음 형식을 따르세요 :

Question 1: Which of the following ... ?
A. ... [correct/incorrect]
B. ... [correct/incorrect]
C. ... [correct/incorrect]
D. ... [correct/incorrect]
Answer: ...

... (반복하여 총 10개)

모든 질문은 “Which of the following ” 형태로 시작해야 하며 "?"로 끝나야 합니다. 선택지는 주어진 데이터 내에서만 생성하시기 바랍니다. 
정답에 해당되는 선택지에는 `[correct]`, 오답에는 `[incorrect]`를 표기하고, 정답 라벨을 `Answer:` 항목에 적어주세요.
'''

In [ ]:
# system_prompt = '''
# You are a question generator for medical examinations. You have to generate 10 multiple-choice questions based on the drug information provided.

# Each question you generate must:
# - Use only the information provided.
# - Begin with “Which of the following”.
# - Immediately follow with “is” or “are”.
# - Use one of the allowed phrase templates listed below to complete the question stem.
# - End with a question mark “?”.
# - Provide exactly four options labeled A., B., C., and D., and tag each option with [correct] or [incorrect].
# - Some questions must allow multiple correct answers (e.g. B/C/D, A/B). Do not make all questions single‑answer only.
# - Distribute correct answers across different option labels; vary which labels (A, B, C, D) are correct across the set of questions so that the correct answers are spread out evenly.
# - Create 10 problems in a row, but each problem must be in the same format as above.

# ### Allowed phrase templates (choose one per question)
# 1. “the … mechanism …” (e.g. “the mechanism of action of X”)
# 2. “the … explanation …” (e.g. “the explanation of how X works”)
# 3. “the most specific etiology …” or “the … etiology …”
# 4. “the … cause of/for …”
# 5. “the … characteristic … that is causing/responsible for …”
# 6. “most likely causing/responsible for …”
# 7. “associated with …” or “consistent with …”
# 8. “contributed to …”
# 9. “the … cause …”
# 10. “(is/are) involved in …”
# 11. “(is/are) prevented by, reducing, or improve …”
# 12. “(is/are) administered to, added to, used to treat, or prescribed to …”
# 13. “(is/are) affected by …”
# 14. “best represents/describes/explains/accounts for …”
# 15. “contributed to …”

# Only use these templates; do not invent new forms. Make sure each of the 10 questions uses a different template and covers a different aspect (e.g., mechanism, explanation, etiology, affected organisms, interactions, toxicity, etc.) so that no topic or pattern is repeated.
# '''


# user_prompt_template = '''
# Drug information:
# - Name: {{ name }}
# {% if mechanism_of_action %}- Mechanism of action: {{ mechanism_of_action }}{% endif %}
# {% if pharmacodynamics %}- Pharmacodynamics: {{ pharmacodynamics }}{% endif %}
# {% if indication %}- Indication: {{ indication }}{% endif %}
# {% if classification %}- Classification: {{ classification }}{% endif %}
# {% if affected_organisms %}- Affected organisms: {{ affected_organisms }}{% endif %}
# {% if toxicity %}- Toxicity: {{ toxicity }}{% endif %}

# Using only the information above, generate 10 multiple-choice questions in English. 
# Each question must strictly follow these rules:

# 1. Start the question with “Which of the following”.
# 2. Immediately use “is” or “are” after the introductory phrase.
# 3. Choose one of the allowed phrase templates from the system instructions to complete the question stem (e.g., “the mechanism of action of { name }”, “the etiology of { condition }”, “associated with { effect }”), and ensure that each of the 10 questions uses a different template and covers a different aspect of the drug (mechanism, explanation, etiology, affected organisms, interactions, toxicity, etc.).
# 4. Provide four answer options labeled A., B., C., and D.; tag each with [correct] or [incorrect]. Construct plausible incorrect options by using other relevant fields from the provided data.
# 5. Some questions must allow multiple correct answers (e.g. B/C/D, A/B). Do not make all questions single‑answer only.
# 6. Distribute correct answers across different option labels; vary which labels (A, B, C, D) are correct across the set of questions so that the correct answers are spread out evenly.
# 7. After listing the options, write “Answer:” followed by the correct option labels separated by “/”.
# 8. Ensure the 10 questions are presented consecutively, each following the exact format below.

# Follow this output format exactly:

# Question 1: Which of the following ... ?
# A. ... [correct/incorrect]  
# B. ... [correct/incorrect]  
# C. ... [correct/incorrect]  
# D. ... [correct/incorrect]  
# Answer: ...

# ... (continue until Question 10)
# '''


In [ ]:
# Jinja2 템플릿을 컴파일
template = jinja2.Template(user_prompt_template)

def render_user_prompt(drug: dict) -> str:
    """주어진 약물 정보를 이용해 Jinja2 템플릿을 렌더링한다."""
    return template.render(
        name=drug.get('name', ''),
        description=drug.get('description', ''),
        mechanism_of_action=drug.get('mechanism_of_action', ''),
        pharmacodynamics=drug.get('pharmacodynamics', ''),
        indication=drug.get('indication', ''),
        classification=drug.get('classification', ''),
        affected_organisms=drug.get('affected_organisms', ''),
        toxicity=drug.get('toxicity', '')
    )

def parse_questions(text: str) -> list:
    """LLM 결과에서 문제를 구조화된 목록으로 파싱."""
    q_pattern = re.compile(r'^Question\s*\d+:\s*(.*)', re.IGNORECASE)
    opt_pattern = re.compile(r'^([ABCD])\.\s*(.*?)\s*\[(?:correct|incorrect)\]', re.IGNORECASE)
    ans_pattern = re.compile(r'^Answer:\s*([A-D](?:[/,][A-D])*)', re.IGNORECASE)
    
    questions = []
    current = {}
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        m_q = q_pattern.match(line)
        if m_q:
            if current:
                questions.append(current)
                current = {}
            current['question'] = m_q.group(1).strip()
            continue
        m_opt = opt_pattern.match(line)
        if m_opt and current is not None:
            label, body = m_opt.groups()
            current[f'option_{label}'] = body.strip()
            continue
        m_ans = ans_pattern.match(line)
        if m_ans and current is not None:
            current['answers'] = m_ans.group(1).replace(' ', '')
            continue
    if current:
        questions.append(current)
    return questions


In [ ]:
def generate_mcqs_for_drug(drug: dict, test_mode: bool = False) -> pd.DataFrame:
    """
    주어진 약물 하나에 대해 LLM을 호출해 10개의 MCQ를 생성하고,
    파싱하여 pandas DataFrame으로 반환합니다.
    """
    user_prompt = render_user_prompt(drug)

    response = client.chat.completions.create(
        model="gpt-4.1",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    # LLM 응답 추출
    llm_output = response.choices[0].message.content.strip()

    if test_mode:
        print("=== 유저 프롬프트 ===")
        print(user_prompt)
        print("="*30)
        print("=== 응답 ===")
        print(llm_output)
        print("="*30)

    # 파싱 후 DataFrame 변환
    parsed = parse_questions(llm_output)
    df = pd.DataFrame(
        parsed,
        columns=["question", "option_A", "option_B", "option_C", "option_D", "answers"]
    )
    return df


# 하나 생성 (테스트)

In [ ]:
# 예시: 첫 번째 약물에 대해 질문 생성
if not drugs:
    print("No drugs loaded. Please check the file path.")
else:
    drug = drugs[0]
    df_questions = generate_mcqs_for_drug(drug, test_mode=True)
    display(df_questions)


In [ ]:
df_questions.to_excel("mcqs_example.xlsx", index=False)

# 전체 생성

In [ ]:
from tqdm import tqdm
import pandas as pd

# 모든 의약품에 대해 객관식 문제 생성하는 메인 함수
def generate_mcqs_for_all_drugs(drugs: list, output_file: str = "all_mcqs.jsonl", checkpoint_file: str = "checkpoint.txt"):
    """
    모든 의약품에 대해 객관식 문제를 생성하고 결과를 저장합니다.
    중단 시에도 이어서 실행할 수 있도록 체크포인트 기능을 포함합니다.
    tqdm으로 진행상황을 볼 수 있습니다.
    JSONL 파일을 저장합니다.
    """
    import json
    from pathlib import Path
    
    # 체크포인트 파일에서 이미 처리된 약물 확인
    processed_drugs = set()
    if Path(checkpoint_file).exists():
        with open(checkpoint_file, 'r', encoding='utf-8') as f:
            for line in f:
                processed_drugs.add(line.strip())
        print(f"이미 처리된 약물 수: {len(processed_drugs)}")
    
    # 결과 파일이 이미 존재하면 기존 결과 로드
    existing_results = []
    if Path(output_file).exists():
        with open(output_file, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    existing_results.append(json.loads(line))
        print(f"기존 결과 로드됨: {len(existing_results)}개")
    
    # 처리할 약물 목록 생성 (이미 처리된 것 제외)
    drugs_to_process = [drug for drug in drugs if drug.get('name') not in processed_drugs]
    print(f"처리할 약물 수: {len(drugs_to_process)}")
    
    if not drugs_to_process:
        print("모든 약물이 이미 처리되었습니다.")
        return existing_results
    
    
    # tqdm을 사용하여 진행상황 표시 (간결하게)
    with tqdm(drugs_to_process, desc="약물 문제 생성", unit="개", ncols=80) as pbar:
        for drug in pbar:
            drug_name = drug.get('name', 'Unknown')
            
            try:
                # 현재 처리 중인 약물명을 postfix로 표시
                pbar.set_postfix_str(f"처리 중: {drug_name[:20]}{'...' if len(drug_name) > 20 else ''}")
                
                # 기존 함수를 사용하여 문제 생성
                df_questions = generate_mcqs_for_drug(drug)
                
                # 각 문제를 개별 JSONL 항목으로 저장
                for _, question_row in df_questions.iterrows():

                    answer_idx = question_row['answers']
                    answer_text = question_row[f'option_{answer_idx}']
                    question_entry = {
                        "question": question_row['question'],
                        "answer": answer_text,  # 정답 텍스트
                        "options": {
                            "A": question_row['option_A'],
                            "B": question_row['option_B'], 
                            "C": question_row['option_C'],
                            "D": question_row['option_D']
                        },
                        "meta_info": drug_name,  # 약물명을 메타 정보로 사용
                        "answer_idx": answer_idx  # 정답 인덱스 (A, B, C, D 중 하나)
                    }
                    
                    # JSONL 파일에 개별 항목으로 저장
                    with open(output_file, 'a', encoding='utf-8') as f:
                        f.write(json.dumps(question_entry, ensure_ascii=False) + '\n')
                
                
                # 체크포인트 업데이트
                with open(checkpoint_file, 'a', encoding='utf-8') as f:
                    f.write(drug_name + '\n')
                
                # 성공 시 postfix 업데이트
                pbar.set_postfix_str(f"완료: {drug_name[:20]}{'...' if len(drug_name) > 20 else ''}")
                
            except Exception as e:
                # 오류 발생 시 postfix에 오류 표시
                pbar.set_postfix_str(f"오류: {drug_name[:15]}{'...' if len(drug_name) > 15 else ''}")
                # 오류가 발생해도 계속 진행
                continue
    
    
    print(f"\n모든 처리 완료! 결과는 {output_file}에 저장되었습니다.")
    return existing_results + [result for result in drugs_to_process if result.get('name') not in processed_drugs]

In [ ]:
# 실행 코드
if __name__ == "__main__":
    # 모든 약물에 대해 문제 생성 실행
    results = generate_mcqs_for_all_drugs(
        drugs=drugs,
        output_file="output/all_mcqs.jsonl",
        checkpoint_file="output/checkpoint.txt"
    )
    
    print(f"총 생성된 약물 수: {len(results)}")

# 결과

In [ ]:
# 결과 확인 및 분석을 위한 헬퍼 함수
def analyze_generated_results(output_file: str = "output/all_mcqs.jsonl"):
    """
    생성된 결과를 분석하고 통계를 출력합니다.
    """
    import json
    from pathlib import Path
    
    if not Path(output_file).exists():
        print(f"결과 파일 {output_file}이 존재하지 않습니다.")
        return
    
    results = []
    with open(output_file, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                results.append(json.loads(line))
    
    print(f"총 생성된 약물 수: {len(results)}")
    
    # 각 약물별 문제 수 확인
    question_counts = [len(result['questions']) for result in results]
    print(f"평균 문제 수: {sum(question_counts) / len(question_counts):.1f}")
    print(f"최소 문제 수: {min(question_counts)}")
    print(f"최대 문제 수: {max(question_counts)}")
    
    # # 문제가 없는 약물 확인
    # empty_results = [r for r in results if len(r['questions']) == 0]
    # if empty_results:
    #     print(f"문제가 생성되지 않은 약물 수: {len(empty_results)}")
    #     for result in empty_results:
    #         print(f"  - {result['drug_name']}")
    
    # return results

# 결과 분석 실행
analyze_generated_results()

In [ ]:
import json
import random
from pathlib import Path
from collections import Counter

def shuffle_options(jsonl_file: str, output_file: str = "all_mcqs_shuffled.jsonl"):
    """
    JSONL 파일의 각 문제에서 선택지를 셔플하고 정답 인덱스를 올바르게 수정합니다.
    원본과 변환 후의 정답 비율을 분석합니다.
    """
    shuffled_questions = []
    original_answer_counts = Counter()
    
    # JSONL 파일 읽기
    with open(jsonl_file, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                question = json.loads(line)
                
                # 원본 정답 인덱스 카운트
                original_answer_counts[question['answer_idx']] += 1
                
                # 현재 정답 텍스트 저장
                correct_answer = question['answer']
                
                # 선택지들을 리스트로 변환
                options_list = [
                    ('A', question['options']['A']),
                    ('B', question['options']['B']),
                    ('C', question['options']['C']),
                    ('D', question['options']['D'])
                ]
                
                # 선택지 셔플
                random.shuffle(options_list)
                
                # 셔플된 선택지로 새로운 options 딕셔너리 생성
                new_options = {}
                new_answer_idx = None
                
                for i, (old_idx, option_text) in enumerate(options_list):
                    new_idx = chr(65 + i)  # A, B, C, D
                    new_options[new_idx] = option_text
                    
                    # 정답인 경우 새로운 인덱스 저장
                    if option_text == correct_answer:
                        new_answer_idx = new_idx
                
                # 수정된 문제 생성
                shuffled_question = {
                    'question': question['question'],
                    'answer': correct_answer,
                    'options': new_options,
                    'meta_info': question['meta_info'],
                    'answer_idx': new_answer_idx
                }
                
                shuffled_questions.append(shuffled_question)
    
    # 셔플된 결과를 새 파일에 저장
    with open(output_file, 'w', encoding='utf-8') as f:
        for question in shuffled_questions:
            f.write(json.dumps(question, ensure_ascii=False) + '\n')
    
    # 셔플 후 정답 인덱스 카운트
    shuffled_answer_counts = Counter()
    for question in shuffled_questions:
        shuffled_answer_counts[question['answer_idx']] += 1
    
    total_questions = len(shuffled_questions)
    
    print(f"총 {total_questions}개 문제의 선택지가 셔플되어 {output_file}에 저장되었습니다.")
    
    # 원본 정답 비율 출력
    print("\n=== 원본 파일 정답 비율 ===")
    for option in ['A', 'B', 'C', 'D']:
        count = original_answer_counts[option]
        percentage = (count / total_questions) * 100
        print(f"{option}: {count}개 ({percentage:.1f}%)")
    
    # 셔플 후 정답 비율 출력
    print("\n=== 셔플 후 정답 비율 ===")
    for option in ['A', 'B', 'C', 'D']:
        count = shuffled_answer_counts[option]
        percentage = (count / total_questions) * 100
        print(f"{option}: {count}개 ({percentage:.1f}%)")
    
    # 셔플 결과 확인 (처음 5개 문제 출력)
    print("\n=== 셔플 결과 샘플 (처음 5개) ===")
    for i, question in enumerate(shuffled_questions[:5]):
        print(f"\n문제 {i+1}:")
        print(f"질문: {question['question'][:100]}...")
        print(f"정답: {question['answer_idx']} - {question['answer']}")
        print(f"선택지: {question['options']}")

# 실행
shuffle_options("output/all_mcqs.jsonl", "output/all_mcqs_shuffled.jsonl")

In [ ]:
import json
import pandas as pd

def jsonl_to_excel(jsonl_file: str, excel_file: str):
    """
    JSONL 파일을 엑셀 파일로 변환합니다.
    """
    excel_data = []
    
    # JSONL 파일 읽기
    with open(jsonl_file, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            if line.strip():
                try:
                    question = json.loads(line)
                    
                    # 엑셀용 행 데이터 생성
                    excel_row = {
                        'drug_name': question['meta_info'],
                        'question': question['question'],
                        'option_A': question['options']['A'],
                        'option_B': question['options']['B'],
                        'option_C': question['options']['C'],
                        'option_D': question['options']['D'],
                        'answer': question['answer'],
                        'answer_idx': question['answer_idx']
                    }
                    excel_data.append(excel_row)
                    
                except json.JSONDecodeError as e:
                    print(f"라인 {line_num}에서 JSON 파싱 오류: {e}")
                    continue
                except KeyError as e:
                    print(f"라인 {line_num}에서 키 오류: {e}")
                    continue
    
    # DataFrame 생성 및 엑셀 파일 저장
    if excel_data:
        df_excel = pd.DataFrame(excel_data)
        df_excel.to_excel(excel_file, index=False)
        print(f"총 {len(excel_data)}개 문제가 {excel_file}에 저장되었습니다.")
        
        # 데이터 확인
        print(f"\n=== 엑셀 파일 구조 ===")
        print(f"컬럼: {list(df_excel.columns)}")
        print(f"행 수: {len(df_excel)}")
        print(f"\n=== 처음 3개 행 샘플 ===")
        print(df_excel.head(3))
        
    else:
        print("변환할 데이터가 없습니다.")

# 실행
# jsonl_to_excel("output/all_mcqs_shuffled.jsonl", "output/all_mcqs_shuffled.xlsx")

## 문제 품질 검사

In [ ]:
import json
import re
import pandas as pd
from collections import Counter

def find_problematic_indices(jsonl_file: str = "output/all_mcqs_shuffled.jsonl", 
                           output_excel: str = "output/problematic_questions.xlsx"):
    """
    JSONL 파일의 문제 품질을 검사하고 문제가 있는 문제들의 인덱스를 반환합니다.
    """
    quality_issues = {
        'duplicate_options': 0,
        'html_tags': 0,
        'missing_options': 0,
        'empty_questions': 0,
        'very_short_questions': 0,
        'very_long_questions': 0,
        'very_short_options': 0,
        'very_long_options': 0,
        'same_question_duplicates': 0,
        'invalid_answer_idx': 0
    }
    
    problematic_questions = []
    problematic_indices = set()  # 문제가 있는 라인 인덱스 저장
    question_texts = set()
    
    # HTML 태그 패턴
    html_pattern = re.compile(r'<[^>]+>|&[a-zA-Z]+;|&[#]\d+;')
    
    with open(jsonl_file, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            if line.strip():
                try:
                    question = json.loads(line)
                    
                    # 1. 선택지 중복 검사
                    options = question['options']
                    option_values = list(options.values())
                    if len(set(option_values)) != len(option_values):
                        quality_issues['duplicate_options'] += 1
                        problematic_indices.add(line_num)
                        problematic_questions.append({
                            'line': line_num,
                            'issue': 'duplicate_options',
                            'question': question['question'],
                            'options': options,
                            'answer_idx': question['answer_idx'],
                            'meta_info': question['meta_info']
                        })
                    
                    # 2. HTML 태그 검사
                    question_text = question['question']
                    if html_pattern.search(question_text):
                        quality_issues['html_tags'] += 1
                        problematic_indices.add(line_num)
                        problematic_questions.append({
                            'line': line_num,
                            'issue': 'html_tags',
                            'question': question_text,
                            'options': options,
                            'answer_idx': question['answer_idx'],
                            'meta_info': question['meta_info']
                        })
                    
                    # 3. 선택지 결측값 검사
                    for option_key, option_value in options.items():
                        if not option_value or option_value.strip() == '':
                            quality_issues['missing_options'] += 1
                            problematic_indices.add(line_num)
                            problematic_questions.append({
                                'line': line_num,
                                'issue': 'missing_options',
                                'option': option_key,
                                'question': question_text,
                                'options': options,
                                'answer_idx': question['answer_idx'],
                                'meta_info': question['meta_info']
                            })
                    
                    # 4. 빈 질문 검사
                    if not question_text or question_text.strip() == '':
                        quality_issues['empty_questions'] += 1
                        problematic_indices.add(line_num)
                        problematic_questions.append({
                            'line': line_num,
                            'issue': 'empty_questions',
                            'question': question_text,
                            'options': options,
                            'answer_idx': question['answer_idx'],
                            'meta_info': question['meta_info']
                        })
                    
                    # 7. 중복 질문 검사
                    if question_text in question_texts:
                        quality_issues['same_question_duplicates'] += 1
                        problematic_indices.add(line_num)
                        problematic_questions.append({
                            'line': line_num,
                            'issue': 'same_question_duplicates',
                            'question': question_text,
                            'options': options,
                            'answer_idx': question['answer_idx'],
                            'meta_info': question['meta_info']
                        })
                    else:
                        question_texts.add(question_text)
                    
                    # 8. 유효하지 않은 answer_idx 검사
                    if question['answer_idx'] not in ['A', 'B', 'C', 'D']:
                        quality_issues['invalid_answer_idx'] += 1
                        problematic_indices.add(line_num)
                        problematic_questions.append({
                            'line': line_num,
                            'issue': 'invalid_answer_idx',
                            'answer_idx': question['answer_idx'],
                            'question': question_text,
                            'options': options,
                            'meta_info': question['meta_info']
                        })
                    
                except json.JSONDecodeError as e:
                    print(f"라인 {line_num}에서 JSON 파싱 오류: {e}")
                    continue
                except KeyError as e:
                    print(f"라인 {line_num}에서 키 오류: {e}")
                    continue
    
    # 결과 출력
    total_questions = len(question_texts) + quality_issues['same_question_duplicates']
    
    print("=== 문제 품질 검사 결과 ===")
    print(f"총 문제 수: {total_questions}")
    print()
    
    print("=== 품질 이슈 통계 ===")
    for issue, count in quality_issues.items():
        if count > 0:
            percentage = (count / total_questions) * 100
            print(f"{issue}: {count}개 ({percentage:.1f}%)")
    
    # 문제가 있는 인덱스 출력
    problematic_indices_list = sorted(list(problematic_indices))
    print(f"\n문제가 있는 라인 인덱스 {len(problematic_indices_list)}개:")
    print(f"제거할 라인들: {problematic_indices_list}")
    
    # 문제가 있는 문제들을 엑셀 파일로 저장
    if problematic_questions:
        # 엑셀용 데이터 준비
        excel_data = []
        for problem in problematic_questions:
            row = {
                'line': problem['line'],
                'issue': problem['issue'],
                'question': problem['question'],
                'option_A': problem['options']['A'],
                'option_B': problem['options']['B'],
                'option_C': problem['options']['C'],
                'option_D': problem['options']['D'],
                'answer_idx': problem['answer_idx'],
                'meta_info': problem['meta_info']
            }
            
            # 특정 이슈에 대한 추가 정보
            if 'option' in problem:
                row['problematic_option'] = problem['option']
            else:
                row['problematic_option'] = ''
                
            excel_data.append(row)
        
        # DataFrame 생성 및 엑셀 파일 저장
        df_problems = pd.DataFrame(excel_data)
        df_problems.to_excel(output_excel, index=False)
        print(f"문제가 있는 {len(problematic_questions)}개 문제가 {output_excel}에 저장되었습니다.")
    
    return quality_issues, problematic_questions, problematic_indices_list

# 실행
issues, problems, problematic_indices = find_problematic_indices(
    "output/all_mcqs_shuffled.jsonl", 
    "output/problematic_questions.xlsx"
)

In [ ]:
def remove_problematic_questions(input_file: str = "output/all_mcqs_shuffled.jsonl",
                               problematic_indices: list = None,
                               output_file: str = "output/all_mcqs_cleaned.jsonl"):
    """
    문제가 있는 문제들을 제거하고 깨끗한 JSONL 파일을 생성합니다.
    """
    if problematic_indices is None:
        print("제거할 문제 인덱스가 제공되지 않았습니다.")
        return 0, 0
    
    problematic_indices_set = set(problematic_indices)
    print(f"제거할 문제 수: {len(problematic_indices_set)}")
    print(f"제거할 라인들: {sorted(problematic_indices_set)}")
    
    # 깨끗한 문제들만 저장
    kept_questions = 0
    removed_questions = 0
    
    with open(input_file, 'r', encoding='utf-8') as f_in, \
         open(output_file, 'w', encoding='utf-8') as f_out:
        
        for line_num, line in enumerate(f_in, 1):
            if line.strip():
                if line_num not in problematic_indices_set:
                    # 문제가 없는 문제는 그대로 저장
                    f_out.write(line)
                    kept_questions += 1
                else:
                    # 문제가 있는 문제는 건너뛰기
                    removed_questions += 1
                    print(f"라인 {line_num} 제거됨")
    
    print(f"\n처리 완료!")
    print(f"보존된 문제 수: {kept_questions}")
    print(f"제거된 문제 수: {removed_questions}")
    print(f"깨끗한 파일이 {output_file}에 저장되었습니다.")
    
    return kept_questions, removed_questions

# 실행 (첫 번째 셀에서 찾은 problematic_indices 사용)
kept, removed = remove_problematic_questions(
    "output/all_mcqs_shuffled.jsonl",
    problematic_indices,  # 첫 번째 셀에서 반환된 값 사용
    "output/all_mcqs_cleaned.jsonl"
)

In [ ]:
jsonl_to_excel("output/all_mcqs_cleaned.jsonl", "output/all_mcqs_cleaned.xlsx")

# 문항 변환

In [ ]:
# 셀 1: 필요한 라이브러리 임포트 및 recognize_and_rewrite 함수 정의
import json
import re
from collections import Counter

# recognize_and_rewrite 함수 정의 (recognize_and_rewrite_medqa.py에서 가져옴)
prefix = r"(?:\. *([\w '\'\-,]*?))?(what|which of the following)(\s[\w\s'\'\-]*?)(is|are|was|should be|should also be|could have|will be|would have|could be|would be|would|could) "
prefix_2 = r"(?:\. *([\w '\'\-,]*?))?(what|which of the following)([\w\s'\'\-]*?) "
patterns = {
        r"(characteristic of )?the most likely diagnosis(.*?)(?:\?|\:)": ['Which of the followings are the most likely diagnoses? There may be one or more correct choices.','Which of the followings are less likely to be the diagnosis? There may be one or more correct choices.','"{} is likely to be the diagnosis in this case", is the statement above true or false? Please answer true/false.','"{} is unlikely to be the diagnosis in this case", is the statement above true or false? Please answer true/false.'],
        r"the (.*?)mechanism(.*?)(?:\?|\:)":['Which of the followings are the most likely underlying mechanisms of this patient\'s condition? There may be one or more correct choices.','Which of the followings are less likely to be the underlying mechanism of this patient\'s condition? There may be one or more correct choices.','"{} is likely to be the underlying mechanism of this patient\'s condition", is the statement above true or false? Please answer true/false.','"{} is unlikely to be the underlying mechanism of this patient\'s condition", is the statement above true or false? Please answer true/false.'],
        r"the (.*?)explanation(.*?)(?:\?|\:)":['Which of the followings are the most likely underlying mechanisms of this patient\'s condition? There may be one or more correct choices.','Which of the followings are less likely to be the underlying mechanism of this patient\'s condition? There may be one or more correct choices.','"{} is likely to be the underlying mechanism of this patient\'s condition", is the statement above true or false? Please answer true/false.','"{} is unlikely to be the underlying mechanism of this patient\'s condition", is the statement above true or false? Please answer true/false.'],
        r"the most specific etiology(.*?)(?:\?|\:)":[],
        r"the (.*?)(immediate|initial|first|next|long-term) step in ((?:the|his|her|this patient's)\s)?(management|therapy|treatment|evaluation|action)(.*?)(?:\?|\:)":['Which of the followings are the most likely next best steps in management? There may be one or more correct choices.','Which of the followings are less likely to be the next best step in management? There may be one or more correct choices.','"{} is likely to be the next best step in management", is the statement above true or false? Please answer true/false.','"{} is unlikely to be the next best step in management", is the statement above true or false? Please answer true/false.'],
        r"the (.*?)(diagnostic test|diagnostic step|step in diagnosis)(.*?)(?:\?|\:)":[],
        r"the (.*?)next step(.*?)(?:\?|\:)":[],
        r"the([\w\s'\'\-]*?) (management|therapy|treatment|intervention|action|recommendation)([\w\s'\'\-]*?)(?:\?|\:)":[],
        r"the (immediate|initial|first|next|long-term) ([\w\s'\'\-]*?)step(.*?)(?:\?|\:)":['Which of the followings are the most likely next best steps in management? There may be one or more correct choices.','Which of the followings are less likely to be the next best step in management? There may be one or more correct choices.','"{} is likely to be the next best step in management", is the statement above true or false? Please answer true/false.','"{} is unlikely to be the next best step in management", is the statement above true or false? Please answer true/false.'],
        r"the([\w\s'\'\-]*?) etiology (.*?)(?:\?|\:)":['Which of the followings are the most likely etiologies of this patient\'s condition? There may be one or more correct choices.','Which of the followings are less likely to be the etiology of this patient\'s condition? There may be one or more correct choices.','"{} is likely to be the etiology of this patient\'s condition", is the statement above true or false? Please answer true/false.','"{} is unlikely to be the etiology of this patient\'s condition", is the statement above true or false? Please answer true/false.'],
        r"(true of )?the (.*?)cause (of|for) (.*?)(?:\?|\:)":['Which of the followings are the most likely causes of the patient\'s condition? There may be one or more correct choices.','Which of the followings are less likely to be the causes of the patient\'s condition? There may be one or more correct choices.','"{} is likely to be the cause of the patient\'s condition", is the statement above true or false? Please answer true/false.','"{} is unlikely to be the cause of the patient\'s condition", is the statement above true or false? Please answer true/false.'],
        r"(a )?(characteristic of )?the (.*?) (that is )?(causing|responsible for) (.*?)(?:\?|\:)":['Which of the followings are the most likely causes of the patient\'s condition? There may be one or more correct choices.','Which of the followings are less likely to be the causes of the patient\'s condition? There may be one or more correct choices.','"{} is likely to be the cause of the patient\'s condition", is the statement above true or false? Please answer true/false.','"{} is unlikely to be the cause of the patient\'s condition", is the statement above true or false? Please answer true/false.'],
        r"(most likely )?(causing|responsible for) (.*?)(?:\?|\:)":['Which of the followings are the most likely causes of the patient\'s condition? There may be one or more correct choices.','Which of the followings are less likely to be the causes of the patient\'s condition? There may be one or more correct choices.','"{} is likely to be the cause of the patient\'s condition", is the statement above true or false? Please answer true/false.','"{} is unlikely to be the cause of the patient\'s condition", is the statement above true or false? Please answer true/false.'],
        r"the (.*?)(predisposing|inciting|precipitating) factor (.*?)(?:\?|\:)":[],
        r"(a|the) (.*?) factor (.*?)(?:\?|\:)":[],
        r"the (.*?)(next|initial) (.*?)pharmacotherapy(.*?)(?:\?|\:)":[],
        r"the (.*?)(pharmacotherapy|drug)(.*?)(?:\?|\:)":[],
        r"(.*?)confirm (the|your) diagnosis(.*?)(?:\?|\:)":[],
        r"(.*?)(associated|consistent) with(.*?)(?:\?|\:)":[],
        r"(.*?)contributed to(.*?)(?:\?|\:)":[],
        r"the([\w\s'\'\-]+?)finding (.*?)(?:\?|\:)":[],
        r"the([\w\s'\'\-]+?) cause(.*?)(?:\?|\:)":[],
        r"(.*?)(seen|found|observed|expected|present )([\w\s'\'\-]*?)(?:\?|\:)":[],
        r"([\w\s'\'\-]*?)true( about)?(.*?)(?:\?|\:)":[],
        r"([\w\s'\'\-]*?)involved in(.*?)(?:\?|\:)":[],
        r"([\w\s'\'\-]*?)(prevented|reducing|improve) (.*?)(?:\?|\:)":[],
        r"([\w\s'\'\-]*?)(administered|added to|used to treat|prescribed to)(.*?)(?:\?|\:)":[],
        r"([\w\s'\'\-]*?)affected by(.*?)(?:\?|\:)":[],
        r"([\w\s'\'\-]+?)(management|therapy|treatment|intervention)([\w\s'\'\-]*?)(?:\?|\:)":[],
    }
patterns_2 = {
        r"(best )?(represents|describes|explains|explain|accounts for) (.*?)(?:\?|\:)":[],
        r"tests (is|are) (required to|indicated to|most likely to) (investigate|determine|elucidate) (.*?)(?:\?|\:)":[],
        r"contributed to (.*?)(?:\?|\:)":[],
    }

def recognize_and_rewrite(ques):
    '''
        recognize the question and rewrite it into multiple questions
    '''
    def rewrite(ans_list, pattern_idx, target_idx):
        neg_word_dict = {
            'is':'is not',
            'are':'are not',
            'were':'were not',
            'was':'was not',
            'should be':'should not be',
            'should also be':'should not be',
            'could have':'could not have',
            'will be':'will not be',
            'would have':'would not have',
            'could be':'could not be',
            'would be':'would not be',
            'would':'would not',
            'could':'could not'
        }
        plu_dict = {k:k for k in neg_word_dict}
        plu_dict['is'] = 'are'
        plu_dict['was'] = 'were'
        tails_list = [
            r'{}the most likely diagnosis{}',
            r'the {}mechanism{}',
            r"the {}explanation{}",
            r"the most specific etiology{}",
            r"the {}{} step in {}{}{}",
            r"the {}{}{}",
            r"the {}next step{}",
            r"the{} {}{}",
            r"the {} {}step{}",
            r"the{} etiology {}",
            r"{}the {}cause {} {}",
            r"{}{}the {} {}{} {}",
            r"{}{} {}",
            r"the {}{} factor {}",
            r"{} {} factor {}",
            r"the {}{} {}pharmacotherapy{}",
            r"the {}{}{}",
            r"{}confirm {} diagnosis{}",
            r"{}{} with{}",
            r"{}contributed to{}",
            r"the{}finding {}",
            r"the{} cause{}",
            r"{}{}{}",
            r"{}true{}{}",
            r"{}involved in{}",
            r"{}{} {}",
            r"{}{}{}",
            r"{}affected by{}",
            r"{}{}{}",
            r"{}{} {}",
            r"tests {} {} {}",
            r"contributed to {}",
        ]
        neg_tails_list = [
            r"do not {}{} {}",
            r"tests {} not {} {} {}",
            r"do not contributed to {}",
        ]
        new_list = []
        for one in range(len(ans_list.regs)):
            if ans_list[one]:
                new_list.append(ans_list[one])
            else:
                new_list.append('')
        ans_list = new_list
        if pattern_idx == 0:
            plu_word = plu_dict[ans_list[4]]
            neg_plu_word = neg_word_dict[plu_word]
            maq = '{}{}{}{} '.format(ans_list[1], ans_list[2], ans_list[3], plu_word)
            neg_maq = '{}{}{}{} '.format(ans_list[1], ans_list[2], ans_list[3], neg_plu_word)
            tfq = '{}[option_text] {} '.format(ans_list[1], ans_list[4])
            neg_tfq = '{}[option_text] {} '.format(ans_list[1], neg_word_dict[ans_list[4]])
            fib = 'Fill in the blank space below with the appropriate content: {}___ {} '.format(ans_list[1], ans_list[4])
        else:
            maq = '{}{}{} '.format(ans_list[1], ans_list[2], ans_list[3])
            neg_maq = '{}{}{} '.format(ans_list[1], ans_list[2], ans_list[3])
            tfq = '{}[option_text] '.format(ans_list[1])
            neg_tfq = '{}[option_text] '.format(ans_list[1])
            fib = 'Fill in the blank space below with the appropriate content: {}___ '.format(ans_list[1])
        tail = tails_list[len(patterns)*pattern_idx+target_idx].format(*ans_list[4:]) if pattern_idx == 1 else tails_list[len(patterns)*pattern_idx+target_idx].format(*ans_list[5:])
        neg_tail = neg_tails_list[target_idx].format(*ans_list[4:]) if pattern_idx == 1 else None
        maq = maq + tail + '? There may be one or more correct choices.'
        neg_maq = neg_maq + tail +'? There may be one or more correct choices.' if pattern_idx == 0 else neg_maq + neg_tail +'? There may be one or more correct choices.'
        tfq_1 = 'Statement: "'+tfq + tail+'.", is the statement above true or false? Please answer true/false.'
        neg_tfq_1 = 'Statement: "'+neg_tfq+tail+'.", is the statement above true or false? Please answer true/false.' if pattern_idx == 0 else 'Statement: "'+neg_tfq+neg_tail+'.", is the statement above true or false? Please answer true/false.'
        tfq_2 = 'Statement: "'+tfq + tail+' comparing with other four options: {}, {}, {}, and {}.", is the statement above true or false? Please answer true/false.'
        neg_tfq_2 = 'Statement: "'+neg_tfq + tail+' comparing with other four options: {}, {}, {}, and {}.", is the statement above true or false? Please answer true/false.' if pattern_idx == 0 else 'Statement: "'+neg_tfq+neg_tail+' comparing with other four options: {}, {}, {}, and {}.", is the statement above true or false? Please answer true/false.'
        
        fib = fib + tail + '.'
        
        return maq, neg_maq, tfq_1, neg_tfq_1, tfq_2, neg_tfq_2, fib
    
    ans_list = None
    ans = []
    target_p = None
    final_phrase = None
    target_idx = -1
    pattern_idx = -1
    ques_list = None
    for i, p in enumerate(patterns):
        if not ans_list:
            ans_list=re.finditer(prefix+p,ques,re.DOTALL|re.I)
            try:
                ans_list = list(ans_list)
                ans_list = ans_list[-1]
            except:
                ans_list = None
            target_idx = i
            pattern_idx = 0
        else:
            break
    if not ans_list:
        for i, p in enumerate(patterns_2):
            if not ans_list:
                ans_list = re.finditer(prefix_2+p,ques,re.DOTALL|re.I)
                try:
                    ans_list = list(ans_list)[-1]
                except:
                    ans_list = None
                target_p = p
                target_idx = i
                pattern_idx = 1
            else:
                break

    ques_list=re.search(prefix,ques,re.DOTALL|re.I)
    if not ques_list:
        ques_list=re.search(prefix_2,ques,re.DOTALL|re.I)

    maq, neg_maq, tfq, neg_tfq, tfq_2, neg_tfq_2, fib = None, None, None,None,None,None,None
    if ans_list:
        ans = ans_list[0]
        maq, neg_maq, tfq, neg_tfq, tfq_2, neg_tfq_2, fib = rewrite(ans_list, pattern_idx, target_idx)

    if not target_p:
        out = []
    elif pattern_idx == 0:
        out = patterns[target_p] 
    else:
        out = patterns_2[target_p] 
    return pattern_idx, target_idx, target_p, ans, out, ans_list, maq, neg_maq, tfq, neg_tfq, tfq_2, neg_tfq_2, fib

In [ ]:
# 셀 2: 데이터 로드 및 변환 실행
# output/all_mcqs_cleaned.jsonl 파일에서 데이터 로드
data = []
with open('output/all_mcqs_cleaned.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line.strip()))

print(f"총 {len(data)}개의 문제를 로드했습니다.")

# 변환 통계
hit = 0
ques_set = set()
counter = []
unmatched_ques = set()
counter_2 = []

# 변환된 문제들을 저장할 리스트
converted_questions = []

for idx, item in enumerate(data):
    ques = item['question']
    options = item['options']
    answer = item['answer_idx']
    
    # recognize_and_rewrite 함수를 사용하여 문제 변환
    pattern_idx, target_idx, target_p, ans_list, out, matchobj, maq, neg_maq, tfq, neg_tfq, tfq_2, neg_tfq_2, fib = recognize_and_rewrite(ques)
    
    if ans_list and len(ans_list) > 0:
        hit += 1
        
        ques_set.add((pattern_idx, target_idx, target_p, ans_list))
        counter.append((pattern_idx, target_idx))

        # 변환된 문제들 생성
        maq_ques = ques.replace(ans_list, '. '+maq)
        neg_maq_ques = ques.replace(ans_list, '. '+neg_maq)
        tfq_ques = ques.replace(ans_list, '. '+tfq)
        neg_tfq_ques = ques.replace(ans_list, '. '+neg_tfq)
        tfq_2_ques = ques.replace(ans_list, '. '+tfq_2)
        neg_tfq_2_ques = ques.replace(ans_list, '. '+neg_tfq_2)
        fib_ques = ques.replace(ans_list, '. '+fib)

        # 변환된 문제들을 딕셔너리로 저장
        converted_item = {
            'original_question': ques,
            'maq_question': maq_ques,
            'neg_maq_question': neg_maq_ques,
            'tfq_question': tfq_ques,
            'neg_tfq_question': neg_tfq_ques,
            'tfq_2_question': tfq_2_ques,
            'neg_tfq_2_question': neg_tfq_2_ques,
            'fib_question': fib_ques,
            'options': options,
            'answer': answer,
            'pattern_idx': pattern_idx,
            'target_idx': target_idx,
            'target_pattern': target_p
        }
        converted_questions.append(converted_item)
    else:
        unmatched_ques.add(ques)

print(f"변환 성공: {hit}개")
print(f"변환 실패: {len(unmatched_ques)}개")
print(f"변환 성공률: {hit/len(data)*100:.2f}%")

In [ ]:
# 셀 3: 변환 결과 확인 및 통계
# 변환된 문제들의 패턴별 통계
pattern_counter = Counter(counter)
print("패턴별 변환 통계:")
for (pattern_idx, target_idx), count in pattern_counter.most_common():
    print(f"패턴 {pattern_idx}, 타겟 {target_idx}: {count}개")

# 변환된 문제 샘플 확인
if converted_questions:
    print("\n=== 변환된 문제 샘플 ===")
    sample = converted_questions[0]
    print(f"원본 문제: {sample['original_question']}")
    print(f"MAQ 변환: {sample['maq_question']}")
    print(f"TFQ 변환: {sample['tfq_question']}")
    print(f"FIB 변환: {sample['fib_question']}")
    print(f"옵션: {sample['options']}")
    print(f"정답: {sample['answer']}")

In [ ]:
# 셀 4: 변환된 문제들을 파일로 저장 (수정된 버전)
# JSONL 형식으로 저장
output_file = 'output/converted_questions.jsonl'
with open(output_file, 'w', encoding='utf-8') as f:
    for item in converted_questions:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print(f"변환된 {len(converted_questions)}개의 문제를 {output_file}에 저장했습니다.")

# 변환 통계를 별도 파일로 저장 (튜플 키를 문자열로 변환)
pattern_stats = {}
for (pattern_idx, target_idx), count in pattern_counter.items():
    pattern_stats[f"pattern_{pattern_idx}_target_{target_idx}"] = count

stats = {
    'total_questions': len(data),
    'converted_questions': hit,
    'failed_questions': len(unmatched_ques),
    'conversion_rate': hit/len(data)*100,
    'pattern_statistics': pattern_stats
}

with open('output/conversion_statistics.json', 'w', encoding='utf-8') as f:
    json.dump(stats, f, ensure_ascii=False, indent=2)

print("변환 통계를 output/conversion_statistics.json에 저장했습니다.")

In [ ]:
# 셀 5: 변환 실패한 문제들 확인 (선택사항)
if unmatched_ques:
    print("=== 변환 실패한 문제들 ===")
    print(f"총 {len(unmatched_ques)}개의 문제가 변환되지 않았습니다.")
    
    # 처음 5개만 출력
    for i, ques in enumerate(list(unmatched_ques)[:5]):
        print(f"{i+1}. {ques}")
        print()
    
    # 실패한 문제들을 별도 파일로 저장
    with open('output/failed_questions.txt', 'w', encoding='utf-8') as f:
        for ques in unmatched_ques:
            f.write(ques + '\n')
    
    print("변환 실패한 문제들을 output/failed_questions.txt에 저장했습니다.")